# 06 — Sensitivity Analysis and Final Outputs

Tests whether the Top 50 remains stable under three plausible weighting schemes:

- **Balanced** — final project model
- **Safety First** — greater emphasis on crashes and vulnerable-road-user injuries
- **Access / Network First** — greater emphasis on schools, subway access, and protected-network endpoint proximity

The final published Top 50 is the **Balanced** scenario. `scenario_hits_v2` records how many of the three Top-50 lists contain each segment.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

import geopandas as gpd

In [ ]:
candidates = gpd.read_file(
    processed / "bike_candidates_ranked_v2_2263.gpkg"
)

# balanced
candidates["score_balanced_v2"] = (
    0.30 * candidates["crash_norm"] +
    0.25 * candidates["vru_norm"] +
    0.15 * candidates["school_norm"] +
    0.10 * candidates["subway_norm"] +
    0.10 * candidates["vz_priority"] +
    0.10 * candidates["connectivity_norm"]
)

# safety first
candidates["score_safety_v2"] = (
    0.40 * candidates["crash_norm"] +
    0.30 * candidates["vru_norm"] +
    0.10 * candidates["school_norm"] +
    0.05 * candidates["subway_norm"] +
    0.05 * candidates["vz_priority"] +
    0.10 * candidates["connectivity_norm"]
)

# access / network first
candidates["score_access_v2"] = (
    0.20 * candidates["crash_norm"] +
    0.15 * candidates["vru_norm"] +
    0.25 * candidates["school_norm"] +
    0.15 * candidates["subway_norm"] +
    0.10 * candidates["vz_priority"] +
    0.15 * candidates["connectivity_norm"]
)

candidates[
    ["score_balanced_v2", "score_safety_v2", "score_access_v2"]
].describe()

In [ ]:
def get_top50(df, score_col):
    return (
        df.sort_values(
            [
                score_col,
                "vru_injury_den_mi",
                "crash_den_mi",
                "school_count_500ft",
                "subway_count_500ft",
                "connectivity_score",
                "SegmentID"
            ],
            ascending=[False, False, False, False, False, False, True]
        )
        .head(50)
        .copy()
    )

top50_balanced = get_top50(candidates, "score_balanced_v2")
top50_safety = get_top50(candidates, "score_safety_v2")
top50_access = get_top50(candidates, "score_access_v2")

balanced_ids = set(top50_balanced["SegmentID"])
safety_ids = set(top50_safety["SegmentID"])
access_ids = set(top50_access["SegmentID"])

print("Balanced ∩ Safety:", len(balanced_ids & safety_ids))
print("Balanced ∩ Access:", len(balanced_ids & access_ids))
print("Safety ∩ Access:", len(safety_ids & access_ids))
print("In all 3:", len(balanced_ids & safety_ids & access_ids))

In [ ]:
candidates["scenario_hits_v2"] = (
    candidates["SegmentID"].isin(balanced_ids).astype(int) +
    candidates["SegmentID"].isin(safety_ids).astype(int) +
    candidates["SegmentID"].isin(access_ids).astype(int)
)

scenario_counts = (
    candidates["scenario_hits_v2"]
    .value_counts()
    .sort_index(ascending=False)
)

print(scenario_counts)

## Final outputs

In [ ]:
robust_v2 = candidates[candidates["scenario_hits_v2"] > 0].copy()
top50_final = top50_balanced.copy()

# ensures the final balanced rank is explicit.
top50_final = (
    top50_final
    .sort_values(
        [
            "score_balanced_v2",
            "vru_injury_den_mi",
            "crash_den_mi",
            "school_count_500ft",
            "subway_count_500ft",
            "connectivity_score",
            "SegmentID"
        ],
        ascending=[False, False, False, False, False, False, True]
    )
    .reset_index(drop=True)
)
top50_final["rank_final"] = np.arange(1, len(top50_final) + 1)

robust_v2.to_file(
    processed / "robust_priority_segments_v2_2263.gpkg",
    driver="GPKG"
)
top50_final.to_file(
    processed / "top50_protected_bike_lane_priority_v2_2263.gpkg",
    driver="GPKG"
)

top50_final.drop(columns="geometry").to_csv(
    tables_dir / "top50_priority_segments_v2.csv",
    index=False
)

sensitivity_summary = pd.DataFrame({
    "scenario_hits": [3, 2, 1, 0],
    "count": [
        int((candidates["scenario_hits_v2"] == 3).sum()),
        int((candidates["scenario_hits_v2"] == 2).sum()),
        int((candidates["scenario_hits_v2"] == 1).sum()),
        int((candidates["scenario_hits_v2"] == 0).sum())
    ]
})
sensitivity_summary.to_csv(
    tables_dir / "sensitivity_summary_v2.csv",
    index=False
)

print("Robust V2 segments:", len(robust_v2))
print("Final Top 50:", len(top50_final))
print("Unique final SegmentIDs:", top50_final["SegmentID"].nunique())
print(sensitivity_summary)

## Final borough distribution

In [ ]:
boro_map = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens"
}

candidate_borough = (
    candidates.groupby("LBoro")
    .size()
    .rename("candidate_count")
    .reset_index()
)
candidate_borough["Borough"] = candidate_borough["LBoro"].map(boro_map)

top50_borough = (
    top50_final.groupby("LBoro")
    .size()
    .rename("top50_count")
    .reset_index()
)
top50_borough["Borough"] = top50_borough["LBoro"].map(boro_map)

borough_compare = (
    candidate_borough[["Borough", "candidate_count"]]
    .merge(
        top50_borough[["Borough", "top50_count"]],
        on="Borough",
        how="left"
    )
)
borough_compare["top50_count"] = borough_compare["top50_count"].fillna(0).astype(int)
borough_compare["candidate_share_pct"] = (
    borough_compare["candidate_count"] /
    borough_compare["candidate_count"].sum() * 100
)
borough_compare["top50_share_pct"] = (
    borough_compare["top50_count"] /
    borough_compare["top50_count"].sum() * 100
)
borough_compare["representation_ratio"] = (
    borough_compare["top50_share_pct"] /
    borough_compare["candidate_share_pct"]
)

borough_compare = borough_compare.sort_values(
    "top50_count", ascending=False
)

borough_compare.to_csv(
    tables_dir / "top50_borough_summary_v2.csv",
    index=False
)

borough_compare.round(2)